In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["OPENAI_API_KEY"] = ""

llm = init_chat_model("gpt-5.2")

In [3]:
from pydantic import BaseModel
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage

class greetState(BaseModel):
    messages: Annotated[list, add_messages]

def chatbotnode(state: greetState):
    response = llm.invoke(state.messages)
    return {"messages": [response]}   # correct reducer usage

graph = StateGraph(greetState)

graph.add_node("chatBot", chatbotnode)
graph.add_edge(START, "chatBot")
graph.add_edge("chatBot", END)

final_graph = graph.compile()

res = final_graph.invoke({
    "messages": [
        HumanMessage(content="my name is raju")
    ]
})

print(res["messages"][-1].content)

Hi Raju — nice to meet you. What would you like help with today?


ADDING MEMORY

In [4]:
from pydantic import BaseModel
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage

class greetState(BaseModel):
    messages: Annotated[list, add_messages]

def chatbotnode(state: greetState):
    response = llm.invoke(state.messages)
    return {"messages": [response]}   # correct reducer usage

graph = StateGraph(greetState)

graph.add_node("chatBot", chatbotnode)
graph.add_edge(START, "chatBot")
graph.add_edge("chatBot", END)

final_graph = graph.compile()

res = final_graph.invoke({
    "messages": [
        HumanMessage(content="what is my name")
    ]
})

print(res["messages"][-1].content)

I don’t know your name from this chat. If you tell me what you’d like to be called, I’ll use it.


In [10]:
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage

class greetState(BaseModel):
    messages: Annotated[list, add_messages]

def chatbotnode(state: greetState):
    response = llm.invoke(state.messages)
    return {"messages": [response]}   # correct reducer usage

graph = StateGraph(greetState)

graph.add_node("chatBot", chatbotnode)
graph.add_edge(START, "chatBot")
graph.add_edge("chatBot", END)

final_graph = graph.compile()
memory_saver = InMemorySaver()
config={"configurable":{"thread_id":"123"}}
res = final_graph.invoke({
    "messages": [
        HumanMessage(content="my name is nihal")
    ]
}, config=config)



In [11]:
print(res["messages"][-1].content)

Hi Nihal — nice to meet you. What would you like help with today?


In [29]:
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage,AIMessage

class greetState(BaseModel):
    messages: Annotated[list, add_messages]

def chatbotnode(state: greetState):
    response = llm.invoke(state.messages)
    return {"messages": [AIMessage(content=response.content)]}   # correct reducer usage

graph = StateGraph(greetState)

graph.add_node("chatBot", chatbotnode)
graph.add_edge(START, "chatBot")
graph.add_edge("chatBot", END)

memory_saver = InMemorySaver()
final_graph = graph.compile(checkpointer=memory_saver)

config={"configurable":{"thread_id":"1234"}}
# STEP 1
final_graph.invoke({
    "messages": [
        HumanMessage(content="My name is nihaal")
    ]
}, config=config)

# STEP 2
res = final_graph.invoke({
    "messages": [
        HumanMessage(content="what is my name")
    ]
}, config=config)

print(res["messages"][-1].content)


Your name is **Nihaal**.


In [32]:
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage

class greetState(BaseModel):
    messages: Annotated[list, add_messages]

# ---- GUARDRAIL FUNCTION ----
def guardrail_check(message: str):
    blocked_words = ["hack", "attack", "illegal"]
    
    for word in blocked_words:
        if word in message.lower():
            return False
    return True

# ---- NODE ----
def chatbotnode(state: greetState):
    user_msg = state.messages[-1].content

    # 🔥 GUARDRAIL CHECK
    if not guardrail_check(user_msg):
        return {"messages": [AIMessage(content="❌ This request is not allowed")]}

    response = llm.invoke(state.messages)
    return {"messages": [AIMessage(content=response.content)]}

# ---- GRAPH ----
graph = StateGraph(greetState)

graph.add_node("chatBot", chatbotnode)
graph.add_edge(START, "chatBot")
graph.add_edge("chatBot", END)

memory_saver = InMemorySaver()
final_graph = graph.compile(checkpointer=memory_saver)

config = {"configurable": {"thread_id": "1234"}}

# STEP 1
final_graph.invoke({
    "messages": [HumanMessage(content="how to hack a system")]
}, config=config)



print(res["messages"][-1].content)

Your name is **Nihaal**.


In [37]:
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage

class greetState(BaseModel):
    messages: Annotated[list, add_messages]

# ---- GUARDRAIL FUNCTION ----
def guardrail_check(message: str):
    blocked_words = ["hack", "attack", "illegal"]
    
    for word in blocked_words:
        if word in message.lower():
            return False
    return True

# ---- NODE ----
def chatbotnode(state: greetState):
    user_msg = state.messages[-1].content

    # 🔥 GUARDRAIL CHECK
    if not guardrail_check(user_msg):
        return {"messages": [AIMessage(content="❌ This request is not allowed")]}

    response = llm.invoke(state.messages)
    return {"messages": [AIMessage(content=response.content)]}

# ---- GRAPH ----
graph = StateGraph(greetState)

graph.add_node("chatBot", chatbotnode)
graph.add_edge(START, "chatBot")
graph.add_edge("chatBot", END)

memory_saver = InMemorySaver()
final_graph = graph.compile(checkpointer=memory_saver)

config = {"configurable": {"thread_id": "1234"}}

# STEP 1
res=final_graph.invoke({
    "messages": [HumanMessage(content="how to hack a system")]
}, config=config)



print(res["messages"][-1].content)

❌ This request is not allowed
